In [2]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import math

# Hyperparameters
NUM_LAYERS = 4
D_MODEL = 128
NUM_HEADS = 4
DFF = 512
DROPOUT_RATE = 0.1
MAX_LEN = 40


In [3]:
sentences = [
    "i love machine learning",
    "transformers are powerful",
    "attention is all you need"
]

tokenizer = tf.keras.preprocessing.text.Tokenizer(filters="")
tokenizer.fit_on_texts(sentences)

vocab_size = len(tokenizer.word_index) + 1
sequences = tokenizer.texts_to_sequences(sentences)
sequences = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=MAX_LEN)


In [ ]:
def positional_encoding(seq_len, d_model):
    angle_rads = np.arange(seq_len)[:, np.newaxis] / np.power(
        10000, (2 * (np.arange(d_model)[np.newaxis, :] // 2)) / d_model
    )

    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    return tf.cast(angle_rads[np.newaxis, ...], tf.float32)

def scaled_dot_product_attention(q, k, v, mask):
    matmul_qk = tf.matmul(q, k, transpose_b=True)
    dk = tf.cast(tf.shape(k)[-1], tf.float32)

    scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

    if mask is not None:
        scaled_attention_logits += (mask * -1e9)

    attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
    output = tf.matmul(attention_weights, v)

    return output


class MultiHeadAttention(layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()

        # d_model must be divisible by num_heads
        # because we split the embedding dimension across heads
        assert d_model % num_heads == 0

        # Number of attention heads
        self.num_heads = num_heads

        # Depth of each head (d_k = d_model / num_heads)
        self.depth = d_model // num_heads

        # Linear projections for Query, Key, Value
        # Each maps input embeddings → d_model
        self.wq = layers.Dense(d_model)
        self.wk = layers.Dense(d_model)
        self.wv = layers.Dense(d_model)

        # Final linear layer after concatenating all heads
        self.dense = layers.Dense(d_model)

    def split_heads(self, x, batch_size):
        """
        Splits the last dimension into (num_heads, depth).
        Then transposes to shape:
        (batch_size, num_heads, seq_len, depth)
        """

        # x shape before: (batch_size, seq_len, d_model)
        x = tf.reshape(
            x, (batch_size, -1, self.num_heads, self.depth)
        )

        # Transpose so attention is computed per head
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, v, k, q, mask):
        """
        v, k, q: input tensors
        mask: padding mask or look-ahead (causal) mask
        """

        # Get dynamic batch size (works for variable batch sizes)
        batch_size = tf.shape(q)[0]

        # Linear projections
        # (batch_size, seq_len, d_model)
        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)

        # Split into multiple heads
        # (batch_size, num_heads, seq_len, depth)
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)

        # Scaled dot-product attention (per head)
        # Output shape: (batch_size, num_heads, seq_len, depth)
        scaled_attention = scaled_dot_product_attention(
            q, k, v, mask
        )

        # Transpose back to prepare for concatenation
        # (batch_size, seq_len, num_heads, depth)
        scaled_attention = tf.transpose(
            scaled_attention, perm=[0, 2, 1, 3]
        )

        # Concatenate heads
        # (batch_size, seq_len, d_model)
        concat_attention = tf.reshape(
            scaled_attention,
            (batch_size, -1, self.num_heads * self.depth)
        )

        # Final linear projection
        return self.dense(concat_attention)

class EncoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super().__init__()

        # Multi-Head Self-Attention
        # Q = K = V = x  (encoder attends to itself)
        self.mha = MultiHeadAttention(d_model, num_heads)

        # Position-wise Feed Forward Network
        # Applied independently to each token
        self.ffn = tf.keras.Sequential([
            layers.Dense(dff, activation='relu'),  # Expand dimension
            layers.Dense(d_model)                  # Project back to d_model
        ])

        # Layer Normalization layers
        # Used after residual connections
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

        # Dropout for regularization during training
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, x, training=False, mask=None):
        """
        x: input embeddings (batch_size, seq_len, d_model)
        mask: padding mask (prevents attending to padding tokens)
        """

        # Self-Attention sub-layer
        # Each token attends to all other tokens in the sequence
        attn_output = self.mha(
            v=x, k=x, q=x, mask=mask
        )

        # Apply dropout (only during training)
        attn_output = self.dropout1(attn_output, training=training)

        # Residual connection + Layer Normalization
        out1 = self.layernorm1(x + attn_output)

        # Feed Forward Network sub-layer
        # Applied position-wise to each token
        ffn_output = self.ffn(out1)

        # Dropout again
        ffn_output = self.dropout2(ffn_output, training=training)

        # Residual connection + Layer Normalization
        return self.layernorm2(out1 + ffn_output)



class DecoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super().__init__()

        # Masked Multi-Head Self-Attention
        # Prevents access to future tokens
        self.mha1 = MultiHeadAttention(d_model, num_heads)

        # Encoder–Decoder (Cross) Attention
        # Decoder queries encoder outputs
        self.mha2 = MultiHeadAttention(d_model, num_heads)

        # Position-wise Feed Forward Network
        self.ffn = tf.keras.Sequential([
            layers.Dense(dff, activation='relu'),
            layers.Dense(d_model)
        ])

        # Layer Normalization layers
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
        self.layernorm3 = layers.LayerNormalization()

    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        """
        x: decoder input embeddings (batch_size, target_seq_len, d_model)
        enc_output: encoder outputs (batch_size, input_seq_len, d_model)
        look_ahead_mask: causal mask (prevents seeing future tokens)
        padding_mask: masks padding tokens from encoder output
        """

        # Masked Self-Attention
        # Each token can only attend to previous tokens
        attn1 = self.mha1(
            v=x, k=x, q=x, mask=look_ahead_mask
        )

        # Residual connection + Layer Normalization
        x = self.layernorm1(x + attn1)

        # Encoder–Decoder (Cross) Attention
        # Queries come from decoder
        # Keys and Values come from encoder
        attn2 = self.mha2(
            v=enc_output, k=enc_output, q=x, mask=padding_mask
        )

        # Residual connection + Layer Normalization
        x = self.layernorm2(x + attn2)

        # Feed Forward Network
        ffn_output = self.ffn(x)

        # Residual connection + Layer Normalization
        return self.layernorm3(x + ffn_output)



In [ ]:
class Transformer(tf.keras.Model):
    def __init__(self, vocab_size):
        super().__init__()

        # Token embedding layer
        # Maps token IDs → dense vectors of size D_MODEL
        self.embedding = layers.Embedding(vocab_size, D_MODEL)

        # Positional Encoding
        # Injects word order information into embeddings
        # Shape: (1, MAX_LEN, D_MODEL)
        self.pos_encoding = positional_encoding(MAX_LEN, D_MODEL)

        # Stack of Encoder Layers
        # NUM_LAYERS identical Transformer encoder blocks
        self.enc_layers = [
            EncoderLayer(D_MODEL, NUM_HEADS, DFF)
            for _ in range(NUM_LAYERS)
        ]

        # Final projection layer
        # Maps hidden states → vocabulary logits
        self.final_layer = layers.Dense(vocab_size)

    def call(self, x, training):
        """
        x: input token IDs (batch_size, seq_len)
        training: boolean flag for dropout behavior
        """

        # Get dynamic sequence length
        seq_len = tf.shape(x)[1]

        # Token Embedding
        # (batch_size, seq_len) → (batch_size, seq_len, D_MODEL)
        x = self.embedding(x)

        # Add Positional Encoding
        # Gives the model information about token order
        x += self.pos_encoding[:, :seq_len, :]

        # Pass through Encoder Stack
        # Each layer performs:
        #   Self-Attention → FFN → Residuals → LayerNorm
        for layer in self.enc_layers:
            x = layer(x, training=training, mask=None)

        # Final Linear Layer
        # (batch_size, seq_len, D_MODEL) → (batch_size, seq_len, vocab_size)
        # Produces logits for each token position
        return self.final_layer(x)




In [14]:
model = Transformer(vocab_size)
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

model.fit(sequences, sequences, epochs=10)


Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - accuracy: 0.0167 - loss: 3.4714
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.9000 - loss: 0.8407
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.9000 - loss: 0.7565
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.9000 - loss: 0.6498
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.9000 - loss: 0.6945
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.9000 - loss: 0.6303
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.9000 - loss: 0.6075
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.9000 - loss: 0.6090
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.9000 - loss: 0.6098
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9000 - loss: 0.6091


In [15]:
sample = tokenizer.texts_to_sequences(["attention is powerful"])
sample = tf.keras.preprocessing.sequence.pad_sequences(sample, maxlen=MAX_LEN)

pred = model(sample, training=False)
pred_ids = tf.argmax(pred, axis=-1)

print(pred_ids)


tf.Tensor(
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0]], shape=(1, 40), dtype=int64)
